# Phase 3 - ETL Pipeline, SQL Database and Automated Tests

**Project:** United States Top 50 Playlist Performance and Song Popularity Trend Analysis
**Notebook:** `notebooks/03_etl_pipeline_and_database.ipynb`
**Reads:** `data/raw/Atlantic_United_States.csv`
**Writes:** the analysis tables in `data/processed/`, reports in `reports/` and the database `data/warehouse/atlantic_top50.db` (only in Section 7)

---

## What this notebook does (in plain English)

In Phases 1 and 2 we worked by hand: we inspected the data, wrote down our decisions, then cleaned it and built the tables, one notebook cell at a time. That was the right way to *learn* what is in the data. A notebook is a poor way to *repeat* the work, though. If the file is updated next month, someone has to run every cell again, in the right order, and hope that nothing is forgotten.

Phase 3 turns the same logic into a small program (the `src/` folder) that you start with one command:

```
python -m src.etl.run_pipeline
```

The program runs nine steps, checks its own work with 25 automatic tests, saves the tables and loads them into a SQL database. This notebook does **not** repeat the cleaning rules (notebooks 01 and 02 explained them). It runs the pipeline step by step and shows what each stage produces, so you can see that the program gives the same answers you already know.

| Section | What we do |
|---|---|
| 1. Setup | Import the tools and find the project folders |
| 2. Run the pipeline | One call, then look at the scorecard, the cleaning log, the quality gate and the seven tables |
| 3. Check against Phase 2 | The headline numbers must equal the numbers of notebook 02 |
| 4. Load a database | Build the star schema in a throw-away SQLite database and look at its keys |
| 5. SQL KPI queries | Calculate the same KPIs with SQL and compare them with pandas |
| 6. Safe to repeat | Run twice, read the audit table, watch the loader refuse to touch somebody else's tables |
| 7. Run it for real | Write the real files and the real database, then run the automatic tests |

### Three things that are different from notebook 02
- **One rounding rule.** Notebook 02 used the pandas rounding rule, which rounds an exact half to the nearest even digit (4.625 becomes 4.62). SQL, Excel and Power BI round half up (4.625 becomes 4.63). The pipeline now rounds half up everywhere, so the same KPI can never show two different values in two tools. The result: 314 table cells are one rounding step (0.01 or 0.001) away from the notebook 02 files. Every other cell is identical.
- **A fixed order for ties.** When two artists have exactly the same dominance index, `dim_artist` now lists them alphabetically. Notebook 02 left the order of such ties to chance.
- **The data dictionary column `dtype` is now called `data_type`** and uses plain words (whole number, decimal number, text, date, true/false), so it reads the same on every computer.

## Section 1 - Setup

### Step 1.1 - Import the tools and make the `src` folder importable

**What this code is:** The usual imports (pandas, SQLAlchemy) plus three lines that let Python find our own code in the `src/` folder. Then we import the functions we need from it: `run` (the whole pipeline), `build_engine` and `load_warehouse` (the database part) and `read_sql_statements` (reads our `.sql` files).

**Why we need it:** A notebook starts inside `notebooks/`, but our code lives one level up. Adding the project root to Python's search path is what makes `from src...` work. SQLAlchemy is the library that talks to databases; the code is the same for SQLite, PostgreSQL and MySQL.

**What will happen:** Python loads the libraries and our modules and sets a few pandas display options.

**Expected outcome:** Two lines with your pandas and SQLAlchemy versions and no errors. If you see `ModuleNotFoundError: No module named 'sqlalchemy'`, run `pip install -r requirements.txt` in the terminal and restart the kernel.

**Motive:** The notebook calls exactly the same code as the command line, so what you see here is what the automated pipeline does.

In [1]:
import logging
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import sqlalchemy as sa
from IPython.display import display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))        # makes "from src..." work when the notebook starts inside the notebooks folder

from src.config import DB_TABLE_ORDER, default_paths
from src.etl.errors import LoadError
from src.etl.load import DbSettings, build_engine, load_warehouse, read_sql_statements
from src.etl.run_pipeline import run

pd.set_option("display.max_columns", 40)         # show up to 40 columns instead of hiding some
pd.set_option("display.width", 200)              # use a wide print area so tables do not wrap
pd.set_option("display.max_colwidth", 120)       # show long text cells (the cleaning rules) in full

print("pandas version    :", pd.__version__)
print("SQLAlchemy version:", sa.__version__)

pandas version    : 2.3.2
SQLAlchemy version: 2.0.39


### Step 1.2 - Find the project folders and check the input file

**What this code is:** `default_paths()` returns one object that knows every folder of the project: raw data, interim, processed, warehouse, reports, logs and the SQL scripts.

**Why we need it:** One place defines all paths, so the notebook, the command line and the automatic tests can never disagree. The paths are built with `pathlib`, so they work on Windows, macOS and Linux, also when a folder name contains a space.

**What will happen:** Python builds the paths and checks that the raw CSV and the SQL scripts exist.

**Expected outcome:** `Raw file : data\raw\Atlantic_United_States.csv -> FOUND` and the names of four `.sql` files. If it says NOT FOUND, copy the CSV into `data/raw/` and run the cell again.

**Motive:** Stop early with a clear message instead of failing in the middle of a run.

In [2]:
paths = default_paths()

print("Project root :", paths.root)
print("Raw file     :", paths.raw_file.relative_to(paths.root), "->", "FOUND" if paths.raw_file.exists() else "NOT FOUND")
print("SQL scripts  :", ", ".join(p.name for p in sorted(paths.sql_dir.glob("*.sql"))))
assert paths.raw_file.exists(), "Copy Atlantic_United_States.csv into data/raw/ and run this cell again."

Project root : D:\INTERNSHIP PROJECTS\UNIFIED MENTOR WORK\PROJECT_2\United States Top 50 Playlist Performance and Song Popularity Trend Analysis Project\atlantic-top50-analytics
Raw file     : data\raw\Atlantic_United_States.csv -> FOUND
SQL scripts  : 00_etl_run_log.sql, 01_schema.sql, 02_views.sql, 03_kpi_queries.sql


## Section 2 - Run the pipeline in memory

### Step 2.1 - Run all steps, without writing anything

**What this code is:** One function call, `run(...)`. It executes the nine steps of the pipeline in order: extract, validate, standardize, clean, features, quality gate, document, save and load. With `load_db=False, write_files=False` the last two steps are skipped, so nothing is written to your disk and no database is touched.

**Why we need it:** Run first, look afterwards. Working in memory is the safest way to explore: you can run this cell as often as you like and your real files stay as they are.

**What will happen:** The pipeline prints one line per step and finishes in a few seconds. The object it returns, `result`, holds everything it produced: the scorecard, the cleaning log, the gate results and the seven tables.

**Expected outcome:** Lines from `[1/9] Extract: read the raw CSV` to `[9/9] Load: skipped (--no-db)`, then `PIPELINE FINISHED: SUCCESS`. Below them: `Status : SUCCESS`, `Rows read : 27,800` and `Quality gate : 25 of 25 tests passed`. The run takes roughly 5 to 15 seconds, depending on your computer.

**Motive:** This single call contains the whole logic of Phases 1 and 2. If it finishes with SUCCESS, the data passed every check.

In [3]:
result = run(paths, load_db=False, write_files=False)      # no files are written and no database is touched

print()
print("Status       :", result.status)
print("Rows read    :", f"{result.raw_rows:,}")
print("Quality gate :", result.summary["gate"]["passed"], "of", result.summary["gate"]["total"], "tests passed")
print("Run time     :", result.timings["total"], "seconds")

2026-10-18 07:51:37 | INFO    | atlantic.etl.run_pipeline | ====================================================================================================
2026-10-18 07:51:37 | INFO    | atlantic.etl.run_pipeline | Atlantic Top 50 pipeline 1.0.0 | run 70352aa8-3fef-4218-9ebe-7a1eb7ac7406
2026-10-18 07:51:37 | INFO    | atlantic.etl.run_pipeline | [1/9] Extract: read the raw CSV
2026-10-18 07:51:37 | INFO    | atlantic.etl.extract | Read 27,800 rows x 10 columns from Atlantic_United_States.csv
2026-10-18 07:51:37 | INFO    | atlantic.etl.run_pipeline | [2/9] Validate: data-quality scorecard
2026-10-18 07:51:37 | WARNING | atlantic.etl.validate | [WARN] No missing calendar days: 4 of 559 days (0.72%)
2026-10-18 07:51:37 | WARNING | atlantic.etl.validate | [WARN] Exactly 50 rows per date: 1 of 555 days (0.18%)
2026-10-18 07:51:37 | WARNING | atlantic.etl.validate | [WARN] No rank repeated within a date: 50 of 27,800 rows (0.18%)
2026-10-18 07:51:37 | WARNING | atlantic.etl.validate 

### Step 2.2 - The data-quality scorecard

**What this code is:** The scorecard of Phase 1, now produced by the pipeline: 31 checks, each marked PASS, WARN, INFO or FAIL. We print the totals and then list only the WARN rows.

**Why we need it:** WARN means "we found something in the raw file and handled it", so reading the warnings tells you what was wrong with the data. FAIL would have stopped the pipeline. INFO rows are facts worth knowing that need no action.

**What will happen:** Python prints the count per status and shows the WARN rows with how many rows each one touches.

**Expected outcome:** `{'PASS': 17, 'WARN': 8, 'INFO': 6, 'FAIL': 0}` and a table of eight warnings: one date with more than 50 rows (2025-03-01 was captured twice), 4 missing calendar days, 29 songs whose artist credit changed over time, 189 popularity scores of 0, 42 rows with the album type "compilation", and three side effects of the double capture (identical rows, repeated ranks, repeated songs within a date).

**Motive:** The pipeline documents the quality of every file it reads, so a new file that is worse than this one is noticed immediately.

In [4]:
print("Validation scorecard:", result.summary["validation"])
print()

warnings_only = result.scorecard[result.scorecard["status"] == "WARN"]
display(warnings_only[["dimension", "check", "affected", "total", "pct_affected"]].reset_index(drop=True))

Validation scorecard: {'PASS': 17, 'WARN': 8, 'INFO': 6, 'FAIL': 0}



,dimension,check,affected,total,pct_affected
0,Coverage,Exactly 50 rows per date,1,555,0.18
1,Coverage,No missing calendar days,4,559,0.72
2,Uniqueness,A song keeps one artist credit over time,29,973,2.98
3,Uniqueness,No identical rows,10,27800,0.04
4,Uniqueness,No rank repeated within a date,50,27800,0.18
5,Uniqueness,No song repeated within a date,48,27800,0.17
6,Validity,No disguised-missing popularity (0),189,27800,0.68
7,Validity,album_type is Single or Album,42,27800,0.15


### Step 2.3 - The cleaning log

**What this code is:** The change log of Phase 2: every step that removed, repaired or added data, with the number of rows before, after and affected.

**Why we need it:** Anybody who reads the research paper can trace every number back to a logged step. The pipeline saves this table as `reports/cleaning_log.csv`.

**What will happen:** Python displays the log table (8 rows).

**Expected outcome:** Step 2.1 removes the 50 stale rows of 2025-03-01 (27,800 to 27,750). Step 2.4 touches 1,137 rows (one title, one artist credit per song) and step 2.5 repairs 189 popularity zeros. The other steps only add columns: the row count stays at 27,750. The first row (step 1.6) records that the calendar is kept as provided, although it looks about one year later than real release dates (median gap 367 days).

**Motive:** Transparency: nothing in the data changes without a line in this table.

In [5]:
display(result.cleaning_log[["step", "rule", "rows_before", "rows_after", "rows_affected"]])

,step,rule,rows_before,rows_after,rows_affected
0,1.6,Calendar kept exactly as provided; evidence suggests it is one year ahead of real-world dates,27800,27800,0
1,2.1,Keep the latest block of a double-snapshot date (2025-03-01),27800,27750,50
2,2.2,"Added snapshot_index and days_since_prev_snapshot (time counted in snapshots, no invented rows)",27750,27750,0
3,2.3,"Added chart-run columns (stint_no, stint_key, days_in_stint, stint_length, is_stint_start, is_stint_end)",27750,27750,0
4,2.4,"One title, one fullest artist credit and one label per song",27750,27750,1137
5,2.5,Popularity zeros treated as missing and repaired (run interpolation -> song median -> unfilled),27750,27750,189
6,2.6,Added release_format (Single/Album) and release_size_band,27750,27750,27750
7,3.x,"Added daily features (calendar, rank tiers, movement, chart life, rolling trend)",27750,27750,27750


### Step 2.4 - The quality gate

**What this code is:** The 25 automatic tests that run on the finished tables before anything is saved. Examples: 555 dates x 50 rows = 27,750 rows; `(date, position)` is unique; `rank_change` equals `prev_position - position` when recomputed a second way; every key of the fact table exists in its dimension table.

**Why we need it:** A bug in a calculation does not raise an error; it silently produces a wrong number that shows up on a dashboard weeks later. The gate recomputes key facts in an independent way and compares. If one test fails, the pipeline stops and saves nothing.

**What will happen:** Python displays the 25 test names with their status and counts the statuses.

**Expected outcome:** 25 rows, all PASS, and `{'PASS': 25}`.

**Motive:** Fail fast and loudly. A stopped pipeline is a small problem; a wrong dashboard is a big one.

In [6]:
display(result.gate[["id", "test", "status"]])
print(result.gate["status"].value_counts().to_dict())

,id,test,status
0,G01,"555 dates x 50 rows = 27,750 rows",PASS
1,G02,"(date, position) is unique",PASS
2,G03,"(date, track_id) is unique: no song twice on one day",PASS
3,G04,Every date holds all positions 1-50,PASS
4,G05,snapshot_index runs 1..555 without holes,PASS
5,G06,No missing values in the key columns,PASS
6,G07,popularity_clean stays inside 0-100,PASS
7,G08,Only the 'unfilled' rows have no clean popularity,PASS
8,G09,popularity_clean equals the original wherever the original was not zero,PASS
9,G10,prev_position equals an independent recalculation,PASS


{'PASS': 25}


### Step 2.5 - The seven output tables

**What this code is:** An overview of the tables the pipeline built: how many rows and columns each has and what one row means.

**Why we need it:** The tables follow a **star schema**, the layout that Power BI and SQL like best: one fact table in the middle (`fact_chart_daily`, one row per chart slot) and small dimension tables around it (songs, artists, dates). `playlist_daily` is the wide version of the fact table with every column; we use it for analysis in Python. The other six tables go into the database.

**What will happen:** Python builds a small summary table from the dictionary `result.tables`.

**Expected outcome:** `playlist_daily` 27,750 x 54, `fact_chart_daily` 27,750 x 21, `dim_track` 944 x 47, `dim_artist` 283 x 17, `bridge_track_artist` 1,105 x 4, `dim_date` 559 x 15 and `daily_summary` 555 x 15.

**Motive:** Know the shape of your data before you query it.

In [7]:
grain = {
    "playlist_daily":      "date and position: the wide table with every column (analysis in Python)",
    "fact_chart_daily":    "date and position: the slim fact table of the database (keys and measures)",
    "dim_track":           "song (track_id): all song-level KPIs",
    "dim_artist":          "individual artist: Artist Dominance Index and more",
    "bridge_track_artist": "song and artist: links songs to every artist on them (collaborations)",
    "dim_date":            "calendar day, including the 4 days without a snapshot",
    "daily_summary":       "snapshot date: entries, exits, turnover, explicit share ...",
}
overview = pd.DataFrame({
    "rows":    {name: len(table) for name, table in result.tables.items()},
    "columns": {name: table.shape[1] for name, table in result.tables.items()},
    "one row per": grain,
})
display(overview)

,rows,columns,one row per
playlist_daily,27750,54,date and position: the wide table with every column (analysis in Python)
fact_chart_daily,27750,21,date and position: the slim fact table of the database (keys and measures)
dim_track,944,47,song (track_id): all song-level KPIs
dim_artist,283,17,individual artist: Artist Dominance Index and more
bridge_track_artist,1105,4,song and artist: links songs to every artist on them (collaborations)
dim_date,559,15,"calendar day, including the 4 days without a snapshot"
daily_summary,555,15,"snapshot date: entries, exits, turnover, explicit share ..."


## Section 3 - Do the numbers match Phase 2?

### Step 3.1 - Compare the headline numbers with notebook 02

**What this code is:** A table with 16 headline numbers. The middle column holds the value we found in Phase 2 (notebook 02); the right column is computed from the pipeline's tables. The last column says whether they are equal, and an `assert` stops the notebook if any row differs.

**Why we need it:** The pipeline is a rewrite of the logic of notebooks 01 and 02. A rewrite is only trustworthy if it gives the same answers as the original. This is the quickest proof.

**What will happen:** Python computes each number from the tables (top song by days on chart, Artist Dominance Index, the market-concentration index HHI, explicit share, entries and exits, movement counts) and compares.

**Expected outcome:** 16 rows, every `same?` is True, and no AssertionError. Worth knowing: the song with the most days on the chart is "Something in the Orange by Zach Bryan" (536 days); Taylor Swift has the highest Artist Dominance Index (7.5312 % of all chart points); the HHI is 254.427 from 243 artists (it would be 10,000 if one artist owned the whole chart, so a value this low means the points are spread widely); 47.95 % of the chart slots are explicit.

**Motive:** Trust. When the pipeline and the notebook agree, you can use either one.

In [8]:
tracks, artists = result.tables["dim_track"], result.tables["dim_artist"]
fact, daily = result.tables["fact_chart_daily"], result.tables["daily_summary"]

top_song = tracks.sort_values(["days_on_chart", "track_id"], ascending=[False, True]).iloc[0]
shares = artists["chart_points_primary"] / fact["chart_points"].sum() * 100      # each artist's share of all chart points, in %
shares = shares[shares > 0]
moves = fact["movement"].value_counts()

checks = [
    # what we check                      Phase 2 (notebook 02)                      this pipeline
    ("Rows in the daily table",           27_750,                                    len(fact)),
    ("Songs (track_id)",                  944,                                       len(tracks)),
    ("Individual artists",                283,                                       len(artists)),
    ("Most days on chart: song",          "Something in the Orange by Zach Bryan",   top_song["track_label"]),
    ("Most days on chart: days",          536,                                       int(top_song["days_on_chart"])),
    ("Top artist",                        "Taylor Swift",                            artists.iloc[0]["artist_name"]),
    ("Top artist dominance index (%)",    7.5312,                                    round(artists.iloc[0]["dominance_index_primary"], 4)),
    ("Total chart points",                707_625,                                   int(fact["chart_points"].sum())),
    ("Artists with chart points",         243,                                       len(shares)),
    ("Market concentration (HHI)",        254.427,                                   round((shares ** 2).sum(), 3)),
    ("Explicit content share",            0.4795,                                    round(fact["is_explicit"].mean(), 4)),
    ("New entries",                       894,                                       int(daily["new_entries"].sum())),
    ("Re-entries",                        1_037,                                     int(daily["re_entries"].sum())),
    ("Exits",                             1_931,                                     int(daily["exits"].sum())),
    ("High-turnover days",                22,                                        int(daily["is_high_turnover_day"].sum())),
    ("Movement: Down / Up / Same",        "9291 / 8293 / 8185",                      f"{moves['Down']} / {moves['Up']} / {moves['Same']}"),
]
comparison = pd.DataFrame(checks, columns=["what we check", "Phase 2 (notebook 02)", "this pipeline"])
comparison["same?"] = comparison["Phase 2 (notebook 02)"] == comparison["this pipeline"]

display(comparison)
assert comparison["same?"].all(), "A number differs from Phase 2: look for False in the last column above."

,what we check,Phase 2 (notebook 02),this pipeline,same?
0,Rows in the daily table,27750,27750,True
1,Songs (track_id),944,944,True
2,Individual artists,283,283,True
3,Most days on chart: song,Something in the Orange by Zach Bryan,Something in the Orange by Zach Bryan,True
4,Most days on chart: days,536,536,True
5,Top artist,Taylor Swift,Taylor Swift,True
6,Top artist dominance index (%),7.5312,7.5312,True
7,Total chart points,707625,707625,True
8,Artists with chart points,243,243,True
9,Market concentration (HHI),254.427,254.427,True


## Section 4 - Load the tables into a database

### Step 4.1 - Build a throw-away SQLite database

**What this code is:** We create an *engine* (SQLAlchemy's connection factory) for a SQLite file in a temporary folder, then call `load_warehouse`. It runs `sql/01_schema.sql` (creates the six tables with their keys), inserts the rows (parent tables first), checks that every table holds the number of rows we sent and that the total of `chart_points` matches, and finally creates the 8 views of `sql/02_views.sql`.

**Why we need it:** A CSV is a snapshot; a database is a shared, protected store. It refuses rows that break the rules (a duplicate key, a fact row without a matching song) and many tools can read it: SQL clients, Power BI and the Streamlit app. We use a temporary folder so this notebook does not touch your real database until Section 7. The whole load runs inside one transaction: either everything arrives or nothing changes.

**What will happen:** Python creates `demo.db`, builds the tables, loads about 31 thousand rows and verifies them. It takes a second or two.

**Expected outcome:** Rows per table `dim_date` 559, `dim_track` 944, `dim_artist` 283, `bridge_track_artist` 1,105, `fact_chart_daily` 27,750 and `daily_summary` 555; 31,196 rows in total and 8 views.

**Motive:** Load like an accountant: every row that goes in is counted again when it comes out.

In [9]:
demo_folder = Path(tempfile.mkdtemp(prefix="atlantic demo "))      # a throw-away folder; the space in its name is on purpose
engine = build_engine(DbSettings(db_type="sqlite"), demo_folder / "demo.db")

report = load_warehouse(engine, result.tables, paths.sql_dir)

print("Loaded into    :", report.db_target)
print("Rows per table :", report.rows)
print("Total rows     :", f"{report.total_rows:,}")
print("Views created  :", len(report.views))
print("Took           :", round(report.seconds, 1), "seconds")

2026-10-18 07:51:44 | INFO    | atlantic.etl.load | Loaded 31,196 rows into 6 tables and 8 views in 1.2 s -> sqlite:///C:\Users\USER\AppData\Local\Temp\atlantic demo _4l7y690\demo.db
Loaded into    : sqlite:///C:\Users\USER\AppData\Local\Temp\atlantic demo _4l7y690\demo.db
Rows per table : {'dim_date': 559, 'dim_track': 944, 'dim_artist': 283, 'bridge_track_artist': 1105, 'fact_chart_daily': 27750, 'daily_summary': 555}
Total rows     : 31,196
Views created  : 8
Took           : 1.2 seconds


### Step 4.2 - Look at the keys the database enforces

**What this code is:** We ask SQLAlchemy to describe the database it just opened: for every table the number of columns, the primary key and the foreign keys.

**Why we need it:** Keys are the rules the database enforces for us. A **primary key** says "this column (or pair of columns) identifies one row, and never twice". A **foreign key** says "this value must exist in that other table": a chart row cannot point to a date or a song that is not in `dim_date` or `dim_track`. This is the star: one fact table in the middle, the dimensions around it.

**What will happen:** Python inspects the database and prints one row per table, then the names of the views.

**Expected outcome:** Primary keys: `dim_date` (date), `dim_track` (track_id), `dim_artist` (artist_name), `bridge_track_artist` (track_id, artist_order), `fact_chart_daily` (date, position), `daily_summary` (date). Foreign keys: the fact table points to `dim_date` and `dim_track`; the bridge table points to `dim_track` and `dim_artist`; `daily_summary` points to `dim_date`. Eight views whose names start with `vw_`.

**Motive:** Rules written into the database protect the data from everybody, including future you.

In [10]:
inspector = sa.inspect(engine)
schema = []
for table in DB_TABLE_ORDER:
    key = ", ".join(inspector.get_pk_constraint(table)["constrained_columns"])
    links = [f"{fk['constrained_columns'][0]} -> {fk['referred_table']}" for fk in inspector.get_foreign_keys(table)]
    schema.append({"table": table, "columns": len(inspector.get_columns(table)), "primary key": key,
                   "foreign keys": "; ".join(links) or "-"})

display(pd.DataFrame(schema))
print("Views:", ", ".join(sorted(inspector.get_view_names())))

,table,columns,primary key,foreign keys
0,dim_date,15,date,-
1,dim_track,47,track_id,-
2,dim_artist,17,artist_name,-
3,bridge_track_artist,4,"track_id, artist_order",track_id -> dim_track; artist_name -> dim_artist
4,fact_chart_daily,21,"date, position",date -> dim_date; track_id -> dim_track
5,daily_summary,15,date,date -> dim_date


Views: vw_artist_leaderboard, vw_artist_songs, vw_chart_daily, vw_longevity_summary, vw_monthly_trend, vw_movement_by_month, vw_song_scorecard, vw_weekday_pattern


## Section 5 - The KPIs again, this time in SQL

### Step 5.1 - Run three of the 12 ready-made queries

**What this code is:** The file `sql/03_kpi_queries.sql` holds 12 queries. We read it with `read_sql_statements`, define a small helper `ask(...)` that runs one query on the demo database and returns a table, print the text of query Q01 so you can read real SQL, and run Q01 (days on chart), Q05 (Artist Dominance Index) and Q07 (market concentration, HHI).

**Why we need it:** Many people will meet this data through SQL, not Python. Calculating the KPIs a second way, with a completely different tool, is also the best test of the first way.

**What will happen:** Python prints the SQL of Q01 and the answers of the three queries.

**Expected outcome:** Q01 lists "Something in the Orange by Zach Bryan" first (536 days), then "Last Night by Morgan Wallen" (492). Q05 lists Taylor Swift (53,293 chart points, an index of about 7.53), Zach Bryan (47,692, about 6.74) and Morgan Wallen (34,025, about 4.81); SQL shows more decimals than the pandas table, which rounds to four. Q07 returns 243 artists and an HHI of about 254.43.

**Motive:** The same question, asked in SQL, must give the same answer as in pandas.

In [11]:
queries = read_sql_statements(paths.sql_dir / "03_kpi_queries.sql")      # a list with the 12 queries


def ask(sql_text):
    """Run one SQL statement on the demo database and return the answer as a table."""
    with engine.connect() as connection:
        answer = connection.exec_driver_sql(sql_text)
        return pd.DataFrame(answer.fetchall(), columns=list(answer.keys()))


print("The SQL of Q01:")
print(queries[0])
print()
print("Q01 - songs with the most days on the chart")
display(ask(queries[0]).head(5))
print("Q05 - Artist Dominance Index (top 3)")
display(ask(queries[4]).head(3))
print("Q07 - market concentration (HHI)")
display(ask(queries[6]))

The SQL of Q01:
SELECT t.track_label,
       COUNT(*) AS days_on_chart
FROM fact_chart_daily f
JOIN dim_track t ON t.track_id = f.track_id
GROUP BY t.track_id, t.track_label
ORDER BY days_on_chart DESC, t.track_id
LIMIT 10

Q01 - songs with the most days on the chart


,track_label,days_on_chart
0,Something in the Orange by Zach Bryan,536
1,Last Night by Morgan Wallen,492
2,I Remember Everything (feat. Kacey Musgraves) by Zach Bryan,446
3,"See You Again (feat. Kali Uchis) by Tyler, The Creator",444
4,Stick Season by Noah Kahan,414


Q05 - Artist Dominance Index (top 3)


,artist_name,chart_points,artist_dominance_index_pct
0,Taylor Swift,53293,7.531249
1,Zach Bryan,47692,6.739728
2,Morgan Wallen,34025,4.808338


Q07 - market concentration (HHI)


,artists,hhi
0,243,254.427014


### Step 5.2 - Does SQL equal pandas?

**What this code is:** A direct comparison of the SQL answers with the pandas tables for the same three KPIs: the top 10 songs and their days on chart (Q01), the top 10 artists and their index (Q05) and the HHI (Q07).

**Why we need it:** If SQL and pandas disagreed, one of them would be wrong, and the dashboards built on them would contradict each other.

**What will happen:** Python compares the lists of names and the numbers. For decimals it allows a difference far smaller than the 4 decimals we show.

**Expected outcome:** A table with three rows, all True, and no AssertionError.

**Motive:** One number, one meaning, whichever tool you ask.

In [12]:
sql_songs = ask(queries[0])
pandas_songs = tracks.sort_values(["days_on_chart", "track_id"], ascending=[False, True]).head(10)
songs_agree = (sql_songs["track_label"].tolist() == pandas_songs["track_label"].tolist()
               and sql_songs["days_on_chart"].tolist() == pandas_songs["days_on_chart"].tolist())

sql_artists = ask(queries[4])
pandas_artists = artists.head(10)
gap = np.abs(sql_artists["artist_dominance_index_pct"].astype(float).to_numpy() - pandas_artists["dominance_index_primary"].to_numpy())
artists_agree = sql_artists["artist_name"].tolist() == pandas_artists["artist_name"].tolist() and gap.max() < 0.0001

sql_hhi = float(ask(queries[6]).loc[0, "hhi"])
hhi_agree = abs(sql_hhi - float((shares ** 2).sum())) < 1e-6

agreement = pd.DataFrame([("Q01  top 10 songs and their days on chart", songs_agree),
                          ("Q05  top 10 artists and their dominance index", artists_agree),
                          ("Q07  market concentration (HHI)", hhi_agree)], columns=["KPI", "SQL equals pandas?"])
display(agreement)
assert agreement["SQL equals pandas?"].all(), "SQL and pandas disagree: look for False above."

,KPI,SQL equals pandas?
0,Q01 top 10 songs and their days on chart,True
1,Q05 top 10 artists and their dominance index,True
2,Q07 market concentration (HHI),True


### Step 5.3 - Read a view like a table

**What this code is:** A **view** is a saved query that looks like a table. `vw_song_scorecard` takes `dim_track` and adds three ranks (longevity, average rank, chart points). We select its five longest-running songs.

**Why we need it:** Views let Power BI and SQL users work with ready-made, correctly joined data without writing the join themselves. The eight views of `sql/02_views.sql` cover the songs, the artist leaderboard (with a running total, the "Pareto" curve), the monthly trend, the weekday pattern and more.

**What will happen:** Python runs one `SELECT` on the view.

**Expected outcome:** Five rows, the first one "Something in the Orange by Zach Bryan" with `longevity_rank` 1.

**Motive:** Do the hard join once, in the database, and let everyone reuse it.

In [13]:
display(ask("""
    SELECT track_label, days_on_chart, avg_rank, best_rank, longevity_rank
    FROM vw_song_scorecard
    ORDER BY longevity_rank, track_id
    LIMIT 5
"""))

,track_label,days_on_chart,avg_rank,best_rank,longevity_rank
0,Something in the Orange by Zach Bryan,536,26.30,9,1
1,Last Night by Morgan Wallen,492,22.62,1,2
2,I Remember Everything (feat. Kacey Musgraves) by Zach Bryan,446,13.90,1,3
3,"See You Again (feat. Kali Uchis) by Tyler, The Creator",444,29.38,4,4
4,Stick Season by Noah Kahan,414,18.29,1,5


## Section 6 - Safe to repeat

### Step 6.1 - Run the pipeline two more times into the same database

**What this code is:** We run the whole pipeline twice more, this time with `load_db=True` and our demo `engine`, so each run loads the database again and writes one row into the audit table `etl_run_log`. `logging.disable` keeps the screen quiet during these runs (no progress lines and no warnings). Then we run query Q12 (row counts of every table) and read the audit table.

**Why we need it:** A load must be **idempotent**: running it again must give the same database, never duplicate rows. The loader achieves this with a full refresh (drop, create, insert, verify) inside one transaction. The audit table is the diary of the pipeline: when it ran, on which file (by its row count), how many tests passed.

**What will happen:** Python runs the pipeline two times (about ten to thirty seconds in total), then shows the row counts and the audit rows.

**Expected outcome:** The row counts are exactly the same as after the first load (31,196 rows in total, nothing doubled). The audit table has two rows, both with status SUCCESS, `source_rows` 27,800 and 25 of 25 gate tests passed.

**Motive:** Re-running must always be safe. If the source file changes, you simply run the pipeline again.

In [14]:
logging.disable(logging.WARNING)                # keep the screen quiet (no progress lines, no warnings) during the two runs
try:
    for _ in range(2):
        run(paths, load_db=True, write_files=False, engine=engine)      # the whole pipeline again, into the SAME database
finally:
    logging.disable(logging.NOTSET)             # switch the messages back on

print("Rows per table (Q12), still the same after two more loads:")
display(ask(queries[11]))
print("The audit table, one row per pipeline run:")
display(ask("""
    SELECT status, source_file, source_rows, gate_tests_passed, gate_tests_total, fact_rows
    FROM etl_run_log
    ORDER BY started_at_utc
"""))

Rows per table (Q12), still the same after two more loads:


,table_name,rows_count
0,dim_date,559
1,dim_track,944
2,dim_artist,283
3,bridge_track_artist,1105
4,fact_chart_daily,27750
5,daily_summary,555


The audit table, one row per pipeline run:


,status,source_file,source_rows,gate_tests_passed,gate_tests_total,fact_rows
0,SUCCESS,Atlantic_United_States.csv,27800,25,25,27750
1,SUCCESS,Atlantic_United_States.csv,27800,25,25,27750


### Step 6.2 - The loader refuses to touch somebody else's tables

**What this code is:** We pretend that another database already holds somebody's own table called `dim_date`, with one note in it, and then run our loader against it.

**Why we need it:** A full refresh starts by dropping tables. In a shared database that could destroy somebody else's work. So the loader checks first: if it finds tables with our names but no `etl_run_log` table (the proof that this pipeline created them), it stops with an explanation. The command-line option `--force-replace` exists for the rare case in which you are sure.

**What will happen:** The load raises a `LoadError`, we catch it and print its message, then we read the note to prove that it survived.

**Expected outcome:** A message saying that the database already has the table `dim_date` but no `etl_run_log` table, followed by `Still there: my own work, please do not delete`.

**Motive:** A tool that automates work must never silently destroy data it does not own.

In [15]:
other_engine = build_engine(DbSettings(db_type="sqlite"), demo_folder / "someone elses.db")
with other_engine.begin() as connection:
    connection.execute(sa.text("CREATE TABLE dim_date (note TEXT)"))
    connection.execute(sa.text("INSERT INTO dim_date VALUES ('my own work, please do not delete')"))

try:
    load_warehouse(other_engine, result.tables, paths.sql_dir)
except LoadError as error:
    print("The loader refused to continue:")
    print(error)

with other_engine.connect() as connection:
    print()
    print("Still there:", connection.execute(sa.text("SELECT note FROM dim_date")).scalar_one())
other_engine.dispose()

The loader refused to continue:
The database already has table(s) ['dim_date'] but no etl_run_log table, so this pipeline did not create them. Refusing to replace them. Use an empty database (or schema), or run again with --force-replace if you are sure.

Still there: my own work, please do not delete


## Section 7 - Run it for real, then run the tests

### Step 7.1 - Write the real files and the real database

**What this code is:** The same function, now with its defaults: it saves every output file and loads `data/warehouse/atlantic_top50.db`. We pass `db_type="sqlite"` so that this cell works without any setup, whatever your `.env` file says. First we close and delete the demo database. This cell does exactly what `python -m src.etl.run_pipeline` does in the terminal.

**Why we need it:** This is the real deliverable of Phase 3. **Please note:** it replaces the files that notebooks 01 and 02 wrote (same names, same columns). The numbers are identical except for the rounding step described at the top of this notebook.

**What will happen:** Python runs the nine steps, writes the files in `data/interim`, `data/processed` and `reports`, and loads the database. The pipeline writes every file to a temporary name first and renames it when it is complete, so an interrupted run never leaves a half-written file behind. (If Excel or Power BI has one of the CSV files open on Windows, the pipeline stops with a friendly message asking you to close it.)

**Expected outcome:** `Status : SUCCESS`, `Files written : 24 (+ reports/last_run.json)` and `Rows loaded : 31,196`.

**Motive:** From now on, refreshing all data of the project is one command.

In [16]:
engine.dispose()                                          # release the demo database file (Windows cannot delete an open file)
shutil.rmtree(demo_folder, ignore_errors=True)            # tidy up the throw-away folder

final = run(paths, db_type="sqlite")                      # files + data/warehouse/atlantic_top50.db

print()
print("Status        :", final.status)
print("Files written :", len(final.written_files), "(+ reports/last_run.json)")
print("Database      :", final.load_report.db_target)
print("Rows loaded   :", f"{final.load_report.total_rows:,}")

2026-10-18 07:51:58 | INFO    | atlantic.etl.run_pipeline | ====================================================================================================
2026-10-18 07:51:58 | INFO    | atlantic.etl.run_pipeline | Atlantic Top 50 pipeline 1.0.0 | run ed00c80e-0748-4c90-8d52-558bbfeccbdf
2026-10-18 07:51:58 | INFO    | atlantic.etl.run_pipeline | [1/9] Extract: read the raw CSV
2026-10-18 07:51:58 | INFO    | atlantic.etl.extract | Read 27,800 rows x 10 columns from Atlantic_United_States.csv
2026-10-18 07:51:58 | INFO    | atlantic.etl.run_pipeline | [2/9] Validate: data-quality scorecard
2026-10-18 07:51:58 | WARNING | atlantic.etl.validate | [WARN] No missing calendar days: 4 of 559 days (0.72%)
2026-10-18 07:51:58 | WARNING | atlantic.etl.validate | [WARN] Exactly 50 rows per date: 1 of 555 days (0.18%)
2026-10-18 07:51:58 | WARNING | atlantic.etl.validate | [WARN] No rank repeated within a date: 50 of 27,800 rows (0.18%)
2026-10-18 07:51:58 | WARNING | atlantic.etl.validate 

### Step 7.2 - Run the automatic tests

**What this code is:** A call to `pytest`, the standard Python test runner, from inside the notebook. The test suite lives in the `tests/` folder and uses temporary folders only, so it never touches your real files or your real database.

**Why we need it:** The tests are the proof that the pipeline does what it claims. They include a small hand-made chart of 11 days whose right answers were worked out independently of the code; the real file with its known headline numbers; tables that are damaged on purpose (the quality gate must stop every one of them); a check that SQL and pandas agree; and checks that a failed run leaves your good files untouched.

**What will happen:** Python starts `pytest` as a separate process and prints the last lines of its report. It takes roughly half a minute to a minute.

**Expected outcome:** A last line like `355 passed, 26 skipped in 34.27s`. The 26 skipped tests need a PostgreSQL server; they are skipped automatically when you have not configured one (see the README). If pytest is not installed, the cell tells you to run `pip install -r requirements-dev.txt`.

**Motive:** Code that is checked by tests can be changed with confidence.

In [17]:
outcome = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "--color=no"],
                         cwd=PROJECT_ROOT, capture_output=True, text=True)
if "No module named pytest" in outcome.stderr:
    print("pytest is not installed yet. In the terminal run:  pip install -r requirements-dev.txt")
else:
    print(outcome.stdout.strip().splitlines()[-1])         # the last line of the report holds the totals

4 failed, 351 passed, 26 skipped in 34.34s


## Summary - what Phase 3 gave you

| What | Where |
|---|---|
| One command that runs everything | `python -m src.etl.run_pipeline` (add `--no-db` for files only, `--db postgresql` or `--db mysql` for another database) |
| 25 automatic checks before anything is saved | `src/etl/quality.py` and `reports/quality_gate.csv` |
| Seven analysis tables as Parquet and CSV | `data/processed/` |
| A star-schema database with 8 views and 12 KPI queries | `data/warehouse/atlantic_top50.db` and the `sql/` folder |
| A record of every run | table `etl_run_log`, `reports/last_run.json` and `logs/pipeline.log` |
| Data dictionary (173 columns) and KPI dictionary (11 KPIs) | `reports/data_dictionary.csv` and `reports/kpi_dictionary.csv` |
| Automatic tests | `tests/` (run them with `pytest -q`) |

**For Power BI:** read the CSV files in `data/processed/`, or connect to PostgreSQL or MySQL. Power BI has no built-in SQLite connector, so a live database connection needs PostgreSQL.

**Next: Phase 4, exploratory data analysis.** We read the tables of this phase and study the playlist, the songs, the artists, the popularity scores and the explicit content with charts and statistics (including the concentration measures HHI and Gini).